# Stage 4 — SMPL pose for SNGS-043 track 166

This notebook runs 4DHumans/HMR2 on one tracked player, then exports `pose_166.npz` for football3d.

Create a Kaggle Dataset containing `SNGS-043/img1/`, the track JSON, and the SMPL `.pkl` before running. Enable a GPU accelerator.

In [ ]:
from pathlib import Path

INPUT = Path('/kaggle/input')

frame_files = [p for p in INPUT.rglob('000001.jpg') if p.parent.name.lower() == 'img1']
track_files = list(INPUT.rglob('SNGS-043_football-player-detection-v9_botsort.json'))
smpl_files = list(INPUT.rglob('basicmodel_m_lbs_10_207_0_v1.1.0.pkl'))

assert frame_files, f'frame folder not found; Kaggle inputs: {list(INPUT.iterdir())}'
assert track_files, f'track JSON not found; Kaggle inputs: {list(INPUT.iterdir())}'
assert smpl_files, f'SMPL model not found; Kaggle inputs: {list(INPUT.iterdir())}'

FRAMES = frame_files[0].parent
TRACK_JSON = track_files[0]
SMPL_SOURCE = smpl_files[0]
TRACK_ID = 166
START, END = 560, 620
WORK = Path('/kaggle/working/football3d_stage4')
CROPS = WORK / 'crops'
OUT = WORK / 'pose_166.npz'
print('frames:', FRAMES)
print('track JSON:', TRACK_JSON)
print('SMPL model:', SMPL_SOURCE)
print('inputs ok:', len(list(FRAMES.glob('*.jpg'))), 'frames')

In [ ]:
# Install the official 4DHumans code and its Python dependencies.
%cd /kaggle/working
!git clone -q https://github.com/shubham-goel/4D-Humans.git
%cd /kaggle/working/4D-Humans
!pip install -q -e .[all]

In [ ]:
# 4DHumans expects the neutral-model filename. The file contents are your downloaded SMPL model.
import shutil, json, cv2, numpy as np
from pathlib import Path

from hmr2.configs import CACHE_DIR_4DHUMANS
fallback = Path.cwd() / 'data' / 'basicModel_neutral_lbs_10_207_0_v1.0.0.pkl'
cache_model = Path(CACHE_DIR_4DHUMANS) / 'data' / 'smpl' / 'SMPL_NEUTRAL.pkl'
fallback.parent.mkdir(parents=True, exist_ok=True)
cache_model.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(SMPL_SOURCE, fallback)
shutil.copy2(SMPL_SOURCE, cache_model)
print('SMPL fallback:', fallback)
print('SMPL cache:', cache_model)

track = json.loads(TRACK_JSON.read_text())
by_frame = {row['frame']: row['boxes'] for row in track['frames']}
CROPS.mkdir(parents=True, exist_ok=True)
written = []
for frame in range(START, END + 1):
    box = next((b for b in by_frame.get(frame, []) if b.get('id') == TRACK_ID), None)
    if box is None:
        continue
    image = cv2.imread(str(FRAMES / f'{frame:06d}.jpg'))
    x1, y1, x2, y2 = map(float, box['xyxy'])
    w, h = x2 - x1, y2 - y1
    pad_x, pad_y = 0.35 * w, 0.20 * h
    x1, y1 = max(0, int(x1 - pad_x)), max(0, int(y1 - pad_y))
    x2, y2 = min(image.shape[1], int(x2 + pad_x)), min(image.shape[0], int(y2 + pad_y))
    crop = image[y1:y2, x1:x2]
    if crop.size:
        path = CROPS / f'{frame:06d}.jpg'
        cv2.imwrite(str(path), crop)
        written.append(frame)
print('cropped', len(written), 'frames:', written[0], 'to', written[-1])

In [ ]:
# Download HMR2 with resumable parallel connections, then run it on the crops.
import sys, torch, shutil, subprocess, tarfile
sys.path.insert(0, '/kaggle/working/4D-Humans')
from hmr2.configs import CACHE_DIR_4DHUMANS
from hmr2.models import load_hmr2, DEFAULT_CHECKPOINT
from hmr2.datasets.vitdet_dataset import ViTDetDataset
from hmr2.utils import recursive_to

cache = Path(CACHE_DIR_4DHUMANS)
archive = cache / 'hmr2_data.tar.gz'
checkpoint = Path(DEFAULT_CHECKPOINT)
url = 'https://www.cs.utexas.edu/~pavlakos/4dhumans/hmr2_data.tar.gz'
cache.mkdir(parents=True, exist_ok=True)

if not checkpoint.exists():
    aria2 = shutil.which('aria2c')
    if aria2:
        subprocess.run([aria2, '--continue=true', '--max-connection-per-server=16',
                        '--split=16', '--min-split-size=1M', '--file-allocation=none',
                        f'--dir={cache}', f'--out={archive.name}', url], check=True)
    else:
        subprocess.run(['wget', '-c', '--show-progress', '-O', str(archive), url], check=True)
    print('Extracting HMR2...')
    # The server may deliver this already decompressed despite the .tar.gz name.
    with tarfile.open(archive, 'r:*') as bundle:
        bundle.extractall(cache)

assert checkpoint.exists(), f'HMR2 checkpoint missing after extraction: {checkpoint}'
print('HMR2 ready:', checkpoint)

model, model_cfg = load_hmr2(DEFAULT_CHECKPOINT)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device).eval()

frames_out, joints_out, vertices_out = [], [], []
for image_path in sorted(CROPS.glob('*.jpg')):
    image = cv2.imread(str(image_path))
    height, width = image.shape[:2]
    boxes = np.array([[0, 0, width, height]], dtype=np.float32)
    dataset = ViTDetDataset(model_cfg, image, boxes)
    batch = recursive_to(next(iter(torch.utils.data.DataLoader(dataset, batch_size=1))), device)
    with torch.no_grad():
        result = model(batch)
    vertices = result['pred_vertices'][0]
    regressor = model.smpl.J_regressor.to(vertices.device)
    joints = torch.einsum('jk,bkv->bjv', regressor, vertices[None])[0]
    frames_out.append(int(image_path.stem))
    joints_out.append(joints.cpu().numpy())
    vertices_out.append(vertices.cpu().numpy())

np.savez_compressed(OUT, frame=np.array(frames_out), joints=np.array(joints_out), vertices=np.array(vertices_out), fps=np.float32(25))
print('wrote', OUT, 'joints', np.array(joints_out).shape, 'vertices', np.array(vertices_out).shape)

In [ ]:
# Download the result from the Kaggle notebook output.
from IPython.display import FileLink
FileLink(str(OUT))